## Phase 3: Prediction Model Building & Rolling Retraining Pipeline
### Step 1 — Lag Target Pair Engineering & Evaluation Matrix Initialization

Creates feature-target pairs for $y_{t+1}$ forecasting and allocates
memory for daily prediction values across 3 retraining scenarios:
- Scenario A: WASSERSTEIN_60 (quarterly-batch control)
- Scenario B: WASSERSTEIN_120 (semester-batch control)
- Scenario C: ADWIN (pure-stream control)

**Anti-Look-Ahead Discipline:**
Target is created via `shift(-1)` so row $t$ pairs with $y_{t+1}$.
Last row is dropped (no future target available).
Warm-up: rows 0-240. Simulation runs from row 241 onward.

#### 1. Configuration & Imports

Reuses feature list from Phase 1. Adds Phase 3 specific parameters:
- `warmup_rows`: 240 (rows 0-240 = 241 rows of initial training)
- `xgb_window`: 250 (XGBoost rolling window on retrain)
- `oselm_hidden`: 100 (OS-ELM hidden layer size)
- `scenario_controls`: maps scenario → global drift signal name

In [1]:
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


@dataclass
class Config:
    data_path: str = "../dataset/processed/jkse_preprocessed.csv"
    output_dir: str = "../dataset/processed"
    features: list = field(default_factory=lambda: [
        "Log_Return", "Vol_20d", "Vol_60d", "EMA_5",
        "BB_Middle", "BB_Upper", "BB_Lower",
        "Momentum_5d", "Momentum_20d",
    ])
    batch_algorithms: list = field(default_factory=lambda: [
        "mysd", "ks", "psi", "wasserstein",
    ])
    stream_algorithms: list = field(default_factory=lambda: [
        "adwin", "kswin", "ph",
    ])
    batch_windows: list = field(default_factory=lambda: [60, 120])
    consensus_threshold: float = 1 / 3
    # Phase 3 specific
    warmup_rows: int = 240
    xgb_window: int = 250
    oselm_hidden: int = 100
    simulation_start: int = 241
    scenario_controls: dict = field(default_factory=lambda: {
        "A": "Global_wasserstein_60",
        "B": "Global_wasserstein_120",
        "C": "Global_adwin",
    })

In [2]:
cfg = Config()

#### 2. Data Loading & Drift Signal Reconstruction

Loads the clean Phase 1 dataset. Rebuilds global drift decision vectors
(from Phase 2 scoreboards) for the 3 scenario controls.
The consensus rule: >= 1/3 of 9 features must fire simultaneously.

In [3]:
df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True)
df = df.sort_index()

print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Date range: {df.index.min().date()}  to  {df.index.max().date()}")

Loaded 3930 rows x 14 columns
Date range: 2010-03-31  to  2026-06-19


##### 2a. Utility: Load global drift signal from Phase 2 scoreboard

Each scoreboard has 9 feature columns. Row is `1.0` (global drift)
if mean fraction of triggered features >= `consensus_threshold`.

In [4]:
def load_global_signal(
    key: str,
    output_dir: str,
    features: list[str],
    threshold: float,
) -> pd.Series:
    path = Path(output_dir) / f"scoreboard_{key}.csv"
    sb = pd.read_csv(path, index_col=0, parse_dates=True)
    consensus = sb[features].mean(axis=1)
    is_drift = (consensus >= threshold).astype(float)
    is_drift[sb[features].isnull().all(axis=1)] = np.nan
    return is_drift

In [5]:
drift_signals: dict[str, pd.Series] = {}
for scenario, signal_name in cfg.scenario_controls.items():
    # Extract algorithm key from signal_name (e.g. 'Global_wasserstein_60' -> 'wasserstein_60')
    key = signal_name.replace("Global_", "", 1)
    s = load_global_signal(key, cfg.output_dir, cfg.features, cfg.consensus_threshold)
    drift_signals[scenario] = s
    n_triggers = int(s.dropna().sum())
    print(f"Scenario {scenario} ({signal_name}): {n_triggers} global drift days")

Scenario A (Global_wasserstein_60): 273 global drift days
Scenario B (Global_wasserstein_120): 312 global drift days
Scenario C (Global_adwin): 36 global drift days


#### 3. Target Engineering & X/y Split

- `Target_Log_Return = Log_Return.shift(-1)` pairs today's features with tomorrow's return
- Drop the last row (NaN target after shift)
- Split into matrix X (9 features) and vector y (Target_Log_Return)

**Edge case guard:** Verify no NaN leakage in simulation zone (row 241+).

In [6]:
df["Target_Log_Return"] = df["Log_Return"].shift(-1)

n_before = len(df)
df.dropna(subset=["Target_Log_Return"], inplace=True)
n_dropped = n_before - len(df)

X = df[cfg.features].copy()
y = df["Target_Log_Return"].copy()

print(f"Rows before shift:       {n_before}")
print(f"Rows dropped (last row): {n_dropped}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print()

# Sanity: no NaN in simulation zone
assert X.iloc[cfg.simulation_start:].isnull().sum().sum() == 0, "NaN in X simulation zone"
assert y.iloc[cfg.simulation_start:].isnull().sum() == 0, "NaN in y simulation zone"
print("No NaN in simulation zone (row 241+): OK")

Rows before shift:       3930
Rows dropped (last row): 1
X shape: (3929, 9)
y shape: (3929,)

No NaN in simulation zone (row 241+): OK


#### 4. Prediction Storage DataFrame

Allocates 7 columns on a DatetimeIndex:
- `y_true`: actual target (for evaluation in Phase 4)
- `Pred_XGB_A`, `Pred_OSELM_A`: Scenario A (WASSERSTEIN_60)
- `Pred_XGB_B`, `Pred_OSELM_B`: Scenario B (WASSERSTEIN_120)
- `Pred_XGB_C`, `Pred_OSELM_C`: Scenario C (ADWIN)

All prediction columns are NaN for rows 0-240 (warm-up),
initialized to `0.0` placeholder from row 241 onward
(will be overwritten by actual predictions during the prequential loop).

In [7]:
pred_columns = []
for scenario in ["A", "B", "C"]:
    pred_columns.append(f"Pred_XGB_{scenario}")
    pred_columns.append(f"Pred_OSELM_{scenario}")

pred_df = pd.DataFrame(index=X.index, columns=["y_true"] + pred_columns, dtype=float)
pred_df["y_true"] = y

# Fill prediction columns with 0.0 from simulation_start onward
pred_df.iloc[: cfg.simulation_start, 1:] = np.nan
pred_df.iloc[cfg.simulation_start:, 1:] = 0.0

print(f"pred_df shape: {pred_df.shape}")
print(f"Columns: {list(pred_df.columns)}")
print(f"Warm-up rows (NaN): {int(pred_df.iloc[:cfg.simulation_start, 1:].isnull().all(axis=1).sum())}")
print(f"Simulation rows (0.0 placeholder): {len(pred_df) - cfg.simulation_start}")

pred_df shape: (3929, 7)
Columns: ['y_true', 'Pred_XGB_A', 'Pred_OSELM_A', 'Pred_XGB_B', 'Pred_OSELM_B', 'Pred_XGB_C', 'Pred_OSELM_C']
Warm-up rows (NaN): 241
Simulation rows (0.0 placeholder): 3688


#### 5. Validation Report

**Required outputs:**
1. Final dimensions of X and y after target-lag trimming
2. Alignment head check: print X and y at row 241 to confirm
   day-t features pair with day-(t+1) target (no look-ahead bias)
3. Confirm drift signals and pred_df are row-aligned

In [8]:
print("=" * 60)
print("PHASE 3 STEP 1 — VALIDATION REPORT")
print("=" * 60)
print()

print("1. Final Matrix Dimensions")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print()

print("2. Alignment Head Check (row 241 = simulation start)")
print(f"   Simulation start date: {X.index[cfg.simulation_start].date()}")
print()
print("   X.iloc[241:244] (features at day t):")
print(X.iloc[cfg.simulation_start : cfg.simulation_start + 3].round(6).to_string())
print()
print("   y.iloc[241:244] (target = Log_Return at t+1):")
y_slice = y.iloc[cfg.simulation_start : cfg.simulation_start + 3]
print(y_slice.to_frame("Target_Log_Return").round(6).to_string())
print()

# Cross-check: y at row t should equal Log_Return at row t+1
t = cfg.simulation_start
match = np.isclose(y.iloc[t], X.iloc[t + 1]["Log_Return"])
print(f"   Cross-check: y[{t}] ({y.iloc[t]:.6f}) == Log_Return[{t+1}] ({X.iloc[t+1]['Log_Return']:.6f})? {'MATCH' if match else 'MISMATCH!'}")
print()

print("3. Drift Signal & pred_df Alignment")
for sc in ["A", "B", "C"]:
    s = drift_signals[sc]
    # Align to trimmed index
    s_aligned = s.reindex(X.index)
    n_sim = int(s_aligned.iloc[cfg.simulation_start:].sum())
    print(f"   Scenario {sc}: {n_sim} drift triggers in simulation zone")
print()
print(f"   pred_df rows: {len(pred_df)}, X rows: {len(X)} — match: {len(pred_df) == len(X)}")

PHASE 3 STEP 1 — VALIDATION REPORT

1. Final Matrix Dimensions
   X shape: (3929, 9)
   y shape: (3929,)

2. Alignment Head Check (row 241 = simulation start)
   Simulation start date: 2011-03-24

   X.iloc[241:244] (features at day t):
            Log_Return   Vol_20d   Vol_60d        EMA_5    BB_Middle     BB_Upper     BB_Lower  Momentum_5d  Momentum_20d
Date                                                                                                                     
2011-03-24    0.015461  0.008640  0.012943  3556.860136  3539.564062  3631.478055  3447.650070     0.035921      0.048943
2011-03-25   -0.001255  0.008676  0.012882  3573.564310  3562.171777  3653.574521  3470.769033     0.031840      0.046411
2011-03-28   -0.001180  0.008611  0.012803  3583.282456  3578.973730  3660.814894  3497.132567     0.023595      0.037473

   y.iloc[241:244] (target = Log_Return at t+1):
            Target_Log_Return
Date                         
2011-03-24          -0.001255
2011-03-25   

#### 6. OS-ELM Class Definition

Online Sequential Extreme Learning Machine.

**Architecture:**
- Single hidden layer with `n_hidden` neurons, sigmoid activation
- Input weights $W_{in}$ randomly initialized from $U(-1, 1)$, never updated
- Bias $b$ randomly initialized from $U(0, 1)$, never updated
- Output weights $\beta$ solved analytically

**Training:**
- `fit(X_init, y_init)`: Batch initialization via pseudo-inverse
- `partial_fit(X_new, y_new)`: Recursive least squares (RLS) update
  using the Sherman-Morrison-Woodbury formula for $O(H^2)$ per step

**Prediction:**
- `predict(X)`: $H = \sigma(X \cdot W_{in} + b)$, output = $H \cdot \beta$

In [9]:
class OSELM:
    """Online Sequential Extreme Learning Machine.

    Parameters
    ----------
    n_inputs : int
        Number of input features.
    n_hidden : int
        Number of hidden layer neurons.
    activation : str
        Activation function ('sigmoid' or 'relu').
    random_state : int or None
        Seed for reproducible random weights.
    """

    def __init__(
        self,
        n_inputs: int = 9,
        n_hidden: int = 100,
        activation: str = "sigmoid",
        random_state: int | None = 42,
    ):
        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.activation = activation
        self.random_state = random_state

        rng = np.random.default_rng(random_state)
        self.W_in = rng.uniform(-1.0, 1.0, size=(n_inputs, n_hidden))
        self.bias = rng.uniform(0.0, 1.0, size=(1, n_hidden))

        self.beta: np.ndarray | None = None
        self.P: np.ndarray | None = None  # RLS covariance matrix
        self._fitted = False

    def _activate(self, H: np.ndarray) -> np.ndarray:
        if self.activation == "sigmoid":
            H = np.clip(H, -500, 500)
            return 1.0 / (1.0 + np.exp(-H))
        if self.activation == "relu":
            return np.maximum(0.0, H)
        raise ValueError(f"Unknown activation: {self.activation}")

    def _hidden(self, X: np.ndarray) -> np.ndarray:
        """Compute hidden layer output matrix H (N x n_hidden)."""
        return self._activate(X @ self.W_in + self.bias)

    def fit(self, X: np.ndarray, y: np.ndarray):
        """Initial batch training via pseudo-inverse."""
        H = self._hidden(X)
        H_pinv = np.linalg.pinv(H)
        self.beta = H_pinv @ y
        # Initialize RLS covariance: P = (H^T H)^{-1}
        HtH = H.T @ H
        self.P = np.linalg.inv(HtH + 1e-8 * np.eye(self.n_hidden))
        self._fitted = True
        return self

    def partial_fit(self, X_new: np.ndarray, y_new: np.ndarray):
        """Online update using recursive least squares.

        Implements the Sherman-Morrison-Woodbury formula for
        efficient O(H^2) per-sample update.
        """
        if not self._fitted:
            return self.fit(X_new, y_new)

        H = self._hidden(X_new)  # (N x H)
        for i in range(H.shape[0]):
            h = H[i : i + 1, :]  # (1 x H)
            y_i = y_new[i : i + 1]

            # Gain vector: K = P @ h^T / (1 + h @ P @ h^T)
            Ph = self.P @ h.T  # (H x 1)
            denom = 1.0 + (h @ Ph).item()
            K = Ph / denom

            # Update beta
            error = y_i - (h @ self.beta).item()
            self.beta = self.beta + (K.flatten() * error)

            # Update covariance: P = (I - K @ h) @ P
            self.P = self.P - K @ (h @ self.P)

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if not self._fitted:
            raise RuntimeError("OSELM not fitted — call .fit() first")
        H = self._hidden(X)
        return H @ self.beta

##### 6a. OS-ELM Smoke Test

Quick sanity check: train on warm-up data, verify shapes converge.

In [10]:
oselm = OSELM(n_inputs=9, n_hidden=cfg.oselm_hidden, activation="sigmoid")

X_warm = X.iloc[: cfg.simulation_start].values
y_warm = y.iloc[: cfg.simulation_start].values

oselm.fit(X_warm, y_warm)
y_pred = oselm.predict(X_warm[:5])

print(f"OS-ELM smoke test passed:")
print(f"  Input shape:  {X_warm.shape}")
print(f"  Beta shape:   {oselm.beta.shape}")
print(f"  P shape:      {oselm.P.shape}")
print(f"  Predict shape: {y_pred.shape}")
print(f"  Sample preds: {y_pred[:3].round(6)}")

# Partial fit test
oselm.partial_fit(X.iloc[241:245].values, y.iloc[241:245].values)
y_pred2 = oselm.predict(X.iloc[241:245].values)
print(f"  After partial_fit (4 rows): {y_pred2[:2].round(6)}")

OS-ELM smoke test passed:
  Input shape:  (241, 9)
  Beta shape:   (100,)
  P shape:      (100, 100)
  Predict shape: (5,)
  Sample preds: [0.00109 0.00109 0.00109]
  After partial_fit (4 rows): [0.001105 0.001105]


#### 7. Persist Phase 3 Step 1 Artifacts

Saves `pred_df`, drift signals, and X/y arrays as CSV for future steps.

In [11]:
# Save prediction DataFrame
pred_path = Path(cfg.output_dir) / "predictions_step1.csv"
pred_df.to_csv(pred_path)
print(f"Saved pred_df: {pred_path} ({pred_path.stat().st_size / 1024:.1f} KB)")

# Save X and y
X.to_csv(Path(cfg.output_dir) / "X_features.csv")
y.to_frame("Target_Log_Return").to_csv(Path(cfg.output_dir) / "y_target.csv")
print(f"Saved X ({X.shape}) and y ({y.shape})")

# Save drift signals (aligned to trimmed index)
drift_aligned = pd.DataFrame(index=X.index)
for sc, s in drift_signals.items():
    drift_aligned[f"drift_{sc}"] = s.reindex(X.index)
drift_aligned.to_csv(Path(cfg.output_dir) / "drift_signals_aligned.csv")
print(f"Saved drift signals ({drift_aligned.shape})")

Saved pred_df: ../dataset/processed/predictions_step1.csv (204.5 KB)
Saved X ((3929, 9)) and y ((3929,))
Saved drift signals ((3929, 3))
